In [ ]:
# =========================================================
# 1. IMPORTS
# =========================================================
import os
import csv
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, classification_report
from scipy.stats import pearsonr

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)

# =========================================================
# 2. GOOGLE DRIVE + SETUP
# =========================================================
from google.colab import drive
drive.mount("/content/drive")

SEED = 42
K = 5
MODEL_NAME = "distilbert-base-uncased"

OUTPUT_DIR = "/content/drive/MyDrive/DistilBERT_SingleStep_ManualKFold"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================================================
# 3. LOAD BRIGHTER DATASET
# =========================================================
print("\nLoading BRIGHTER dataset from Hugging Face...")

train_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="train"
)

val_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="dev"
)

test_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="test"
)

train_df = train_data.to_pandas()
val_df = val_data.to_pandas()
test_df = test_data.to_pandas()

print("\nOriginal split sizes:")
print("Train:", len(train_df))
print("Dev  :", len(val_df))
print("Test :", len(test_df))

print("\nSample data:")
print(train_df.head())

# =========================================================
# 4. LABEL CONFIGURATION
# =========================================================
EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]
LEVELS = [1, 2, 3]

LABELS = [
    f"{emotion}_{level}"
    for emotion in EMOTIONS
    for level in LEVELS
]

NUM_LABELS = len(LABELS)

print("\nLabels:")
print(LABELS)
print("Number of labels:", NUM_LABELS)

# =========================================================
# 5. CONVERT LABELS TO SINGLE-STEP FORMAT
# =========================================================
def convert_single_step_labels(df):
    df = df.copy()

    if "disgust" in df.columns:
        df = df.drop(columns=["disgust"])

    for emotion in EMOTIONS:
        df[f"{emotion}_1"] = (df[emotion] == 1).astype(int)
        df[f"{emotion}_2"] = (df[emotion] == 2).astype(int)
        df[f"{emotion}_3"] = (df[emotion] == 3).astype(int)

    return df


train_single = convert_single_step_labels(train_df)
val_single = convert_single_step_labels(val_df)
test_single = convert_single_step_labels(test_df)

print("\nSingle-step sample:")
print(train_single[["text"] + LABELS].head())

# =========================================================
# 6. CREATE CV DATA AND KEEP TEST SEPARATE
# =========================================================
cv_df = pd.concat(
    [train_single, val_single],
    ignore_index=True
)

cv_df = cv_df[["text"] + LABELS]

cv_df = cv_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

test_single = test_single[["text"] + LABELS].reset_index(drop=True)

print("\nCross-validation setup:")
print({
    "train_plus_dev_for_cv": len(cv_df),
    "official_test": len(test_single)
})

test_texts = test_single["text"].tolist()

# =========================================================
# 7. TOKENIZATION SETUP
# =========================================================
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

# =========================================================
# 8. ADD LABEL VECTOR + HF DATASET FUNCTION
# =========================================================
def add_labels(example):
    example["labels"] = [
        float(example[label])
        for label in LABELS
    ]

    return example


def prepare_hf_dataset(df):
    dataset = Dataset.from_pandas(
        df,
        preserve_index=False
    )

    dataset = dataset.map(
        tokenize_function,
        batched=True
    )

    dataset = dataset.map(add_labels)

    dataset.set_format(
        type="torch",
        columns=[
            "input_ids",
            "attention_mask",
            "labels"
        ]
    )

    return dataset


test_single_ds = prepare_hf_dataset(test_single)

# =========================================================
# 9. CREATE MANUAL FOLDS FUNCTION
# =========================================================
def create_manual_folds(cv_df, k=5, seed=42):
    n = len(cv_df)

    indices = np.arange(n)

    np.random.seed(seed)
    np.random.shuffle(indices)

    fold_sizes = np.full(k, n // k, dtype=int)
    fold_sizes[:n % k] += 1

    folds = []
    current = 0

    for fold_size in fold_sizes:
        start = current
        stop = current + fold_size

        folds.append(indices[start:stop])

        current = stop

    fold_data = {}

    for fold in range(k):
        fold_number = fold + 1

        val_idx = folds[fold]

        train_idx = np.concatenate([
            folds[i]
            for i in range(k)
            if i != fold
        ])

        fold_train_df = cv_df.iloc[train_idx].reset_index(drop=True)
        fold_val_df = cv_df.iloc[val_idx].reset_index(drop=True)

        fold_data[fold_number] = {
            "train_df": fold_train_df,
            "val_df": fold_val_df
        }

    return fold_data


fold_data = create_manual_folds(
    cv_df=cv_df,
    k=K,
    seed=SEED
)

# =========================================================
# 10. PRINT TRAIN AND VAL CLASS COUNTS SIDE BY SIDE
# =========================================================
def print_fold_class_counts(fold_number):
    fold_train_df = fold_data[fold_number]["train_df"]
    fold_val_df = fold_data[fold_number]["val_df"]

    train_counts = fold_train_df[LABELS].sum().astype(int)
    val_counts = fold_val_df[LABELS].sum().astype(int)

    counts_df = pd.DataFrame({
        "train_count": train_counts,
        "val_count": val_counts
    })

    print("\n================================")
    print(f"Fold {fold_number}")
    print("================================")
    print("Train samples:", len(fold_train_df))
    print("Validation samples:", len(fold_val_df))

    print("\nTrain and validation class counts:")
    print(counts_df)

    return counts_df

# =========================================================
# 11. PLOT SELECTED FOLD CLASS DISTRIBUTION
# =========================================================
def plot_selected_fold(fold_number):
    fold_train_df = fold_data[fold_number]["train_df"]
    fold_val_df = fold_data[fold_number]["val_df"]

    train_counts = [
        int(fold_train_df[label].sum())
        for label in LABELS
    ]

    val_counts = [
        int(fold_val_df[label].sum())
        for label in LABELS
    ]

    x = np.arange(len(LABELS))
    width = 0.35

    plt.figure(figsize=(16, 7))

    train_bars = plt.bar(
        x - width / 2,
        train_counts,
        width,
        label="Train"
    )

    val_bars = plt.bar(
        x + width / 2,
        val_counts,
        width,
        label="Validation"
    )

    for bar in train_bars:
        height = bar.get_height()

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            str(int(height)),
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90
        )

    for bar in val_bars:
        height = bar.get_height()

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            str(int(height)),
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90
        )

    plt.xlabel("Emotion-intensity labels")
    plt.ylabel("Number of samples")

    plt.title(
        f"Fold {fold_number} Class Distribution\n"
        f"Train={len(fold_train_df)}, Validation={len(fold_val_df)}"
    )

    plt.xticks(
        x,
        LABELS,
        rotation=45
    )

    plt.legend()
    plt.tight_layout()
    plt.show()
    plt.close()

# =========================================================
# 12. DISPLAY ALL FOLDS BEFORE TRAINING
# =========================================================
def show_all_folds():
    for fold_number in range(1, K + 1):
        print_fold_class_counts(fold_number)
        plot_selected_fold(fold_number)


show_all_folds()

# =========================================================
# 13. METRICS
# =========================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    f1_macro = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        labels,
        preds,
        average="micro",
        zero_division=0
    )

    pearsons = []

    for i in range(labels.shape[1]):
        if np.std(labels[:, i]) == 0 or np.std(probs[:, i]) == 0:
            pearsons.append(0.0)
        else:
            p, _ = pearsonr(labels[:, i], probs[:, i])
            pearsons.append(0.0 if np.isnan(p) else float(p))

    pearson_mean = float(np.mean(pearsons))

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": pearson_mean
    }